In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, DistilBertForTokenClassification, AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# %% Define the DistilBERT model class for NER
class DistilbertNER(nn.Module):
    """
    NN class based on DistilBERT pretrained model.
    Inputs:
      - tokens_dim : int specifying the dimension of the classifier
    """
    
    def __init__(self, tokens_dim):
        super(DistilbertNER, self).__init__()
        
        if type(tokens_dim) != int:
            raise TypeError('Please tokens_dim should be an integer')

        if tokens_dim <= 0:
            raise ValueError('Classification layer dimension should be at least 1')

        # Load pretrained DistilBERT for token classification
        self.pretrained = DistilBertForTokenClassification.from_pretrained(
            "distilbert-base-cased", num_labels=tokens_dim
        )

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Forward computation of the network
        """
        if labels is None:
            return self.pretrained(input_ids=input_ids, attention_mask=attention_mask)
        return self.pretrained(input_ids=input_ids, attention_mask=attention_mask, labels=labels)



In [2]:
# %% Custom dataset class for NER using DistilBERT
class NerDataset(Dataset):
    """
    Custom dataset class for NER tasks.
    """

    def __init__(self, dataframe, tokenizer, label_map, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        # Ensure the 'Ingredient_Phrases' column is a string
        text = str(row['Ingredient_Phrases'])

        labels = row['Labels']

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            return_attention_mask=True,
            is_split_into_words=False
        )

        # Convert labels to match tokenization
        label_ids = [-100] * self.max_len  # Default padding label
        word_ids = encoding.word_ids()

        labels_list = eval(labels)  # Convert label string back to list

        # Assign NER labels for each word
        for i, word_id in enumerate(word_ids):
            if word_id is None or word_id >= len(labels_list):
                label_ids[i] = -100
            else:
                label_ids[i] = self.label_map.get(labels_list[word_id], -100)

        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        label_ids = torch.tensor(label_ids)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_ids
        }

# %% Label mapping for the NER task
label_map = {
    'QUANTITY': 0,
    'UNIT': 1,
    'NAME': 2,
    'FORM': 3,
    'STATE': 4,
    'DRY/FRESH': 5,
    'SIZE': 6,
    'PREPROCESSING': 7,
    'O': -100  # Outside token or padding
}

        

In [3]:
from sklearn.model_selection import train_test_split
df = pd.read_csv('/home/ritisha21089/NEW_Annotated_Dataset_NER.csv')

# %% Load the new dataset and tokenizer
df = pd.read_csv('/home/ritisha21089/NEW_Annotated_Dataset_NER.csv')
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
#data_loader = DataLoader(ner_dataset, batch_size=16, shuffle=True)

train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
test_dataset = NerDataset(dataframe=test_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# %% Instantiate the model
model = DistilbertNER(tokens_dim=len(label_map))

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# %% Training loop (basic example)
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training settings
epochs = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

/home/ritisha21089/.local/lib/python3.8/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


DistilbertNER(
  (pretrained): DistilBertForTokenClassification(
    (distilbert): DistilBertModel(
      (embeddings): Embeddings(
        (word_embeddings): Embedding(28996, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (transformer): Transformer(
        (layer): ModuleList(
          (0-5): 6 x TransformerBlock(
            (attention): MultiHeadSelfAttention(
              (dropout): Dropout(p=0.1, inplace=False)
              (q_lin): Linear(in_features=768, out_features=768, bias=True)
              (k_lin): Linear(in_features=768, out_features=768, bias=True)
              (v_lin): Linear(in_features=768, out_features=768, bias=True)
              (out_lin): Linear(in_features=768, out_features=768, bias=True)
            )
            (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            

In [5]:

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        # Extract the input ids, attention masks, and labels from the batch
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# %% Evaluation (example metrics calculation)
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        logits = logits.detach().cpu().numpy()
        label_ids = labels.cpu().numpy()

        for i in range(len(label_ids)):
            pred_labels = []
            true_label = []
            for j in range(len(label_ids[i])):
                if label_ids[i][j] != -100:
                    true_label.append(label_ids[i][j])
                    pred_label = np.argmax(logits[i][j])
                    pred_labels.append(pred_label)
            predictions.append(pred_labels)
            true_labels.append(true_label)


Epoch 1/8, Loss: 0.2856
Epoch 2/8, Loss: 0.1061
Epoch 3/8, Loss: 0.0716
Epoch 4/8, Loss: 0.0505
Epoch 5/8, Loss: 0.0408
Epoch 6/8, Loss: 0.0307
Epoch 7/8, Loss: 0.0242
Epoch 8/8, Loss: 0.0206


In [6]:
# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

Precision: 0.8937
Recall: 0.9392
F1-Score: 0.9142


In [9]:
!pip install seqeval scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable


In [10]:
# %% Ensure the required libraries are installed


# %% Import necessary libraries for evaluation
from seqeval.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

# %% Function to evaluate the model and collect predictions
def evaluate_and_inspect(model, data_loader, tokenizer, label_map_inverse, device):
    model.eval()
    predictions = []
    true_labels = []
    sentences = []  # To store the actual ingredient phrases

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # Move logits and labels to CPU
            logits = logits.detach().cpu().numpy()
            label_ids = labels.cpu().numpy()

            # Iterate over each sentence
            for i in range(len(label_ids)):
                pred_labels = []
                true_label = []
                for j in range(len(label_ids[i])):
                    if label_ids[i][j] != -100:  # Ignore padding labels
                        true_label.append(label_map_inverse[label_ids[i][j]])
                        pred_label = np.argmax(logits[i][j])
                        pred_labels.append(label_map_inverse[pred_label])

                # Append predictions and true labels
                predictions.append(pred_labels)
                true_labels.append(true_label)
                # Decode and store original sentence
                sentences.append(tokenizer.decode(input_ids[i], skip_special_tokens=True))

    return predictions, true_labels, sentences